In [1]:
import pandas as pd
import sqlite3

print("SQLite Ready")

SQLite Ready


In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../pharma_analytics.db')

print("Database Created")

OperationalError: unable to open database file

In [3]:
import sqlite3

conn = sqlite3.connect('pharma_analytics.db')

print("Database Created")

Database Created


In [4]:
import pandas as pd

doctors = pd.read_csv('../01_Dataset/raw_data/doctors.csv')
products = pd.read_csv('../01_Dataset/raw_data/products.csv')
medical_reps = pd.read_csv('../01_Dataset/raw_data/medical_reps.csv')
sales = pd.read_csv('../01_Dataset/raw_data/sales.csv')
prescriptions = pd.read_csv('../01_Dataset/raw_data/prescriptions.csv')

print("All CSV Files Loaded")

All CSV Files Loaded


In [5]:
print(doctors.shape)
print(products.shape)
print(medical_reps.shape)
print(sales.shape)
print(prescriptions.shape)

(500, 6)
(50, 4)
(100, 5)
(50000, 8)
(20000, 5)


In [6]:
doctors.to_sql('doctors', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
medical_reps.to_sql('medical_reps', conn, if_exists='replace', index=False)
sales.to_sql('sales', conn, if_exists='replace', index=False)
prescriptions.to_sql('prescriptions', conn, if_exists='replace', index=False)

print("All Tables Loaded Successfully")

All Tables Loaded Successfully


In [7]:
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

print(cursor.fetchall())

[('doctors',), ('products',), ('medical_reps',), ('sales',), ('prescriptions',)]


In [8]:
query = """
SELECT
    p.product_name,
    SUM(s.revenue) AS total_revenue
FROM sales s
JOIN products p
ON s.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_revenue DESC
LIMIT 10
"""

pd.read_sql_query(query, conn)

,product_name,total_revenue
0,OrthoFlex,12490172.88
1,MindCare,12078719.22
2,FlexiCare,11815044.74
3,DiaPlus,11678731.70
4,SkinPrime,11674952.50
5,SkinShield,11467527.18
6,ArteryFlow,11096950.95
7,GlowCare,10975176.16
8,HeartWell,10956465.92
9,SkinPlus,10190653.40


In [9]:
query = """
SELECT
    d.doctor_name,
    d.specialization,
    SUM(p.prescription_count) AS total_prescriptions
FROM prescriptions p
JOIN doctors d
ON p.doctor_id = d.doctor_id
GROUP BY d.doctor_name, d.specialization
ORDER BY total_prescriptions DESC
LIMIT 10
"""

pd.read_sql_query(query, conn)

,doctor_name,specialization,total_prescriptions
0,Melissa Harris,Orthopedic,1011
1,Amy Kelly,Gynecologist,983
2,Kenneth Wolfe,General Physician,918
3,Dr. Randall Allen III,Dermatologist,914
4,Alicia Richardson,Orthopedic,910
5,Dennis Farrell,General Physician,908
6,Thomas Taylor,Cardiologist,908
7,Julie Briggs,Gynecologist,900
8,Barbara Collins,General Physician,875
9,William Pace,Neurologist,875


In [10]:
query = """
SELECT
    m.mr_name,
    m.region,
    SUM(s.revenue) AS total_revenue
FROM sales s
JOIN medical_reps m
ON s.mr_id = m.mr_id
GROUP BY m.mr_name, m.region
ORDER BY total_revenue DESC
LIMIT 10
"""

pd.read_sql_query(query, conn)

,mr_name,region,total_revenue
0,Jeffrey Nelson,Hyderabad,6875567.84
1,Shane Lopez,Chennai,6826659.57
2,Karen Bell,Chennai,6483484.90
3,Samantha Green,Chennai,6361376.86
4,Cynthia Myers,Chennai,6354785.74
5,Dawn Rose,Delhi,6261964.27
6,Antonio Walker,Chennai,6230755.20
7,Donna Charles,Bangalore,6180251.95
8,Susan Ruiz,Delhi,6033274.21
9,Chad Coleman,Chennai,6028535.10


In [11]:
query = """
SELECT
    substr(sale_date,1,7) AS month,
    ROUND(SUM(revenue),2) AS total_revenue
FROM sales
GROUP BY month
ORDER BY month
"""

pd.read_sql_query(query, conn)

,month,total_revenue
0,2024-06,10948713.16
1,2024-07,15134584.35
2,2024-08,14874218.53
3,2024-09,14792304.84
4,2024-10,14532922.57
5,2024-11,14384435.72
6,2024-12,14334081.13
7,2025-01,14821996.80
8,2025-02,13204467.32
9,2025-03,15722040.46


In [12]:
query = """
SELECT
    m.region,
    ROUND(SUM(s.revenue),2) AS total_revenue
FROM sales s
JOIN medical_reps m
ON s.mr_id = m.mr_id
GROUP BY m.region
ORDER BY total_revenue DESC
"""

pd.read_sql_query(query, conn)

,region,total_revenue
0,Chennai,84563809.09
1,Delhi,81019756.17
2,Bangalore,58673192.02
3,Hyderabad,50461877.22
4,Pune,42720310.60
5,Mumbai,33734493.59


In [14]:
query = """
SELECT
    d.specialization,
    SUM(p.prescription_count) AS total_prescriptions,
    ROUND(SUM(s.revenue),2) AS total_revenue
FROM prescriptions p
JOIN doctors d
ON p.doctor_id = d.doctor_id
JOIN sales s
ON p.product_id = s.product_id
GROUP BY d.specialization
ORDER BY total_revenue DESC
"""

pd.read_sql_query(query, conn)

,specialization,total_prescriptions,total_revenue
0,Neurologist,51767077,2.337950e+10
1,Gynecologist,51231061,2.327795e+10
2,Diabetologist,44181103,2.062179e+10
3,Cardiologist,42538294,1.940698e+10
4,General Physician,42460960,1.938512e+10
5,Orthopedic,41274716,1.880807e+10
6,Dermatologist,35658226,1.588744e+10
